# Notebook 2: LoRA & QLoRA Supervised Fine-Tuning

**Goal:** Fine-tune Qwen2.5-0.5B with LoRA/QLoRA on instruction data. Understand every hyperparameter and train a model end-to-end.

**Topics covered:**
- Loading a small model with 4-bit quantization (bitsandbytes)
- Configuring LoRA with PEFT (r, alpha, target_modules, dropout)
- SFTTrainer from TRL with training arguments
- Loss logging with wandb/tensorboard
- Hyperparameter deep-dive (lr, batch_size, grad_accum, warmup)
- Memory optimization (gradient checkpointing, 4-bit vs 8-bit, flash attention)
- Loss curve plotting
- Before/after generation comparison
- Troubleshooting guide (OOM, convergence issues)

In [ ]:
# Cell 1: Install dependencies (CPU-friendly — use a small model)
!pip install transformers accelerate peft trl bitsandbytes datasets wandb tensorboard torch -q

In [ ]:
# Cell 2: Imports and environment check
import os
import json
import gc
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset, load_dataset

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("Running on CPU. Training will be slow but functional with a small model.")

print(f"Transformers: {transformers.__version__}")

plt.style.use('ggplot')

## Part 1: Understanding LoRA — Theory & Math

**LoRA (Low-Rank Adaptation)** freezes the base model weights and injects trainable low-rank matrices into attention layers.

For a weight matrix $W \in \mathbb{R}^{d \times k}$, LoRA decomposes the update as:

$$\Delta W = B \cdot A \quad \text{where} \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}$$

So the forward pass becomes: $h = Wx + \frac{\alpha}{r} \cdot BAx$

**Key hyperparameters (explained below):**
- **r (rank):** The bottleneck dimension. Typically 8–64. Higher = more capacity but more params.
- **alpha:** Scaling factor. The update is scaled by alpha/r. Typically 16–32.
- **target_modules:** Which layers to apply LoRA to. For Qwen/Llama, typically `["q_proj", "k_proj", "v_proj", "o_proj"]`.
- **lora_dropout:** Dropout on the LoRA path. 0.05-0.1 typical.

**QLoRA** adds 4-bit quantization on top of LoRA for even more memory savings.

In [ ]:
# Cell 3: Load Qwen2.5-0.5B with 4-bit quantization

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — better than standard 4-bit
    bnb_4bit_compute_dtype=torch.float16,   # Compute dtype for matmul
    bnb_4bit_use_double_quant=True,         # Double quantization saves ~0.4 bits/param
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 4: Prepare model for k-bit training and apply LoRA

# Necessary step for quantized models — enables gradient checkpointing compatibility
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # Rank: controls capacity of the adapter
    lora_alpha=32,                 # Alpha: scaling factor (effective lr = alpha/r)
    target_modules=[               # Which linear layers to adapt
        "q_proj",                  # Query projection
        "k_proj",                  # Key projection
        "v_proj",                  # Value projection
        "o_proj",                  # Output projection
        "gate_proj",               # Gate in MLP (optional, adds params)
        "up_proj",                 # Up projection in MLP
        "down_proj",               # Down projection in MLP
    ],
    lora_dropout=0.05,             # Dropout on LoRA path
    bias="none",                   # Don't train biases
    task_type=TaskType.CAUSAL_LM,  # Causal language modeling
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Show what % of the total parameters are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / Total: {total:,} = {100 * trainable / total:.2f}%")

## Part 2: Prepare the Training Data

We format our data using Qwen2.5's chat template for SFT.

In [ ]:
# Cell 5: Load and format training data

# Load a small subset for demo purposes
dataset = load_dataset("tatsu-lab/alpaca", split="train[:2000]")
print(f"Loaded {len(dataset)} examples")

# Format function using Qwen2.5 chat template
def format_alpaca_for_qwen(example):
    """Convert Alpaca (instruction, input, output) to Qwen chat format."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
    ]
    # Build user message
    user_content = example["instruction"]
    if example.get("input") and example["input"].strip():
        user_content += f"\n\n{example['input']}"
    messages.append({"role": "user", "content": user_content})
    messages.append({"role": "assistant", "content": example["output"]})

    # Apply chat template to get formatted string
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Apply formatting
dataset = dataset.map(format_alpaca_for_qwen)
print(f"\nFirst formatted example (first 300 chars):")
print(dataset[0]["text"][:300])
print(f"\nTotal tokens in first example: ~{len(tokenizer.encode(dataset[0]['text']))}")

In [ ]:
# Cell 6: Train/val split and response-only collator

# Split
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

# Response template for completion-only loss masking
# This ensures we only compute loss on assistant responses, not user prompts
response_template = "<|im_start|>assistant"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)
print(f"Response template: {repr(response_template)}")
print("Data collator will mask loss on everything before and including the template.")

## Part 3: Training Configuration — Hyperparameter Deep Dive

| Parameter | Typical Range | What It Does |
|-----------|--------------|--------------|
| **learning_rate** | 1e-4 to 5e-4 | LoRA can use higher LRs than full fine-tune because only small matrices are learned |
| **per_device_train_batch_size** | 1–8 | Limited by VRAM. Use grad accumulation to simulate larger batches |
| **gradient_accumulation_steps** | 1–16 | Effective batch_size = per_device * grad_accum * num_gpus |
| **warmup_ratio** | 0.03–0.1 | Linear warmup from 0 to peak LR over this fraction of steps |
| **num_train_epochs** | 1–5 | LoRA converges fast; 1-3 epochs usually enough |
| **weight_decay** | 0.0–0.1 | L2 regularization; often 0 for LoRA |
| **lr_scheduler_type** | "cosine" or "linear" | Cosine with warmup is a good default |
| **max_seq_length** | 512–2048 | Truncate longer sequences to save memory |
| **gradient_checkpointing** | True | Trades compute for memory; ~30% slower but ~50% less VRAM |
| **fp16/bf16** | True | Mixed precision training; bf16 preferred on Ampere+, fp16 on older GPUs |

In [ ]:
# Cell 7: Training arguments

output_dir = "./qwen-lora-sft-output"

training_args = TrainingArguments(
    # ------ Output & Logging ------
    output_dir=output_dir,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,                    # Log loss every 10 steps
    report_to="none",                    # Change to "wandb" or "tensorboard" as needed
    save_steps=200,                      # Save checkpoint every 200 steps
    save_total_limit=2,                  # Keep only the last 2 checkpoints

    # ------ Training ------
    num_train_epochs=2,                  # 1-3 epochs is typical for LoRA
    per_device_train_batch_size=2,       # Small batch for memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # Effective batch = 2 * 4 = 8
    learning_rate=2e-4,                  # LoRA-friendly LR
    warmup_ratio=0.05,                   # 5% of steps for warmup
    lr_scheduler_type="cosine",          # Cosine decay with warmup
    weight_decay=0.01,                   # Light regularization
    optim="adamw_torch",                 # Use PyTorch AdamW (no bitsandbytes dep)

    # ------ Memory ------
    fp16=torch.cuda.is_available(),      # Mixed precision (GPU only)
    gradient_checkpointing=True,         # Trades compute for VRAM

    # ------ Evaluation ------
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ------ Other ------
    seed=42,
    dataloader_num_workers=0,            # 0 to avoid multiprocessing issues in notebooks
    remove_unused_columns=False,
)

print("Training arguments configured.")
print(f"Effective batch size: {training_args.per_device_train_batch_size} * "
      f"{training_args.gradient_accumulation_steps} = "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: ~{len(train_data) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

In [ ]:
# Cell 8: Set up SFTTrainer and run training

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=1024,                 # Truncate long sequences
    dataset_text_field="text",           # Column with formatted text
    packing=False,                       # Set True for higher throughput (combine short seqs)
)

print("Starting training...")
train_result = trainer.train()
print("Training complete!")

In [ ]:
# Cell 9: Extract and plot training/eval loss

# Collect logged metrics
log_history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
if train_steps:
    ax.plot(train_steps, train_losses, alpha=0.5, color="steelblue", label="Train Loss (per step)")
    # Smoothed training loss
    if len(train_losses) > 5:
        window = min(5, len(train_losses) // 4)
        smoothed = np.convolve(train_losses, np.ones(window)/window, mode="valid")
        smooth_steps = train_steps[window-1:]
        ax.plot(smooth_steps[:len(smoothed)], smoothed, color="darkblue", linewidth=2, label=f"Train Loss (smoothed, w={window})")
if eval_steps:
    ax.plot(eval_steps, eval_losses, "o-", color="coral", linewidth=2, label="Eval Loss", markersize=6)

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training & Evaluation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=100, bbox_inches="tight")
plt.show()

# Final metrics
if train_losses:
    first_loss = train_losses[0]
    last_loss = train_losses[-1]
    print(f"First train loss: {first_loss:.4f}")
    print(f"Final train loss: {last_loss:.4f}")
    print(f"Loss reduction: {(first_loss - last_loss) / first_loss * 100:.1f}%")
if eval_losses:
    print(f"Final eval loss: {eval_losses[-1]:.4f}")
    print(f"Best eval loss: {min(eval_losses):.4f}")

In [ ]:
# Cell 10: Save the LoRA adapter

adapter_path = "./qwen-lora-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")

# Show adapter file sizes
import os
for f in sorted(os.listdir(adapter_path)):
    fpath = os.path.join(adapter_path, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

## Part 4: Before vs After — Generation Comparison

In [ ]:
# Cell 11: Test generation — BEFORE fine-tuning (base model)
# We reload the base model without LoRA for a clean comparison

test_prompts = [
    "Explain what machine learning is in simple terms.",
    "Write a Python function to check if a string is a palindrome.",
    "What are three healthy breakfast ideas?",
]

print("Loading base model for comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

def generate_response(model, tok, prompt, max_new_tokens=128):
    """Generate a response for a given prompt."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt")
    if device == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tok.pad_token_id,
        )
    response = tok.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

print("\n" + "=" * 60)
print("BASE MODEL (BEFORE FINE-TUNING)")
print("=" * 60)
for i, prompt in enumerate(test_prompts):
    print(f"\n[Prompt {i+1}]: {prompt}")
    response = generate_response(base_model, base_tokenizer, prompt)
    print(f"[Response]: {response[:300]}")

# Free base model memory
del base_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Cell 12: Test generation — AFTER LoRA fine-tuning

print("=" * 60)
print("LoRA FINE-TUNED MODEL (AFTER)")
print("=" * 60)
for i, prompt in enumerate(test_prompts):
    print(f"\n[Prompt {i+1}]: {prompt}")
    response = generate_response(model, tokenizer, prompt)
    print(f"[Response]: {response[:300]}")

## Part 5: Memory Optimization Deep Dive

In [ ]:
# Cell 13: Memory footprint comparison — 4-bit vs 8-bit vs full precision

print("=" * 60)
print("MEMORY FOOTPRINT COMPARISON")
print("=" * 60)
print(f"""
| Configuration              | Approx VRAM (0.5B model) |
|----------------------------|--------------------------|
| FP32 (full precision)       | ~2 GB                    |
| FP16 (half precision)       | ~1 GB                    |
| INT8 (8-bit quantization)   | ~0.5 GB                  |
| NF4 (4-bit quantization)    | ~0.25 GB                 |
| NF4 + LoRA (our setup)      | ~0.35 GB                 |
""")

# Show current memory usage
current_mem = model.get_memory_footprint() / 1e9
print(f"Current model memory: {current_mem:.3f} GB")
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"CUDA allocated: {allocated:.3f} GB")
    print(f"CUDA reserved: {reserved:.3f} GB")

print("""
Memory-Saving Techniques Used:
  - 4-bit NF4 quantization: ~4x reduction vs FP16
  - Double quantization: Additional ~0.4 bits/param savings
  - Gradient checkpointing: Trades ~30% compute for ~50% VRAM savings
  - LoRA: Only train ~2-5% of parameters (updates stored in FP16)
  - Flash Attention (if available): O(n) memory vs O(n^2) for attention
""")

In [ ]:
# Cell 14: Flash Attention 2 check

print("Checking Flash Attention 2 availability...")
try:
    import flash_attn
    print(f"flash-attn version: {flash_attn.__version__}")
    print("Flash Attention 2 is available! Enable it with:")
    print("  model = AutoModelForCausalLM.from_pretrained(..., attn_implementation='flash_attention_2')")
except ImportError:
    print("Flash Attention 2 is NOT installed.")
    print("To install (Linux, CUDA): pip install flash-attn --no-build-isolation")
    print("On macOS (Apple Silicon): pip install flash-attn-mps (experimental)")
    print("Or use SDPA (sdpa) which is built into PyTorch >= 2.0:")
    print("  model = AutoModelForCausalLM.from_pretrained(..., attn_implementation='sdpa')")

# Check if SDPA is available (built into PyTorch >= 2.0)
if hasattr(torch.nn.functional, 'scaled_dot_product_attention'):
    print("\nSDPA (Scaled Dot-Product Attention) is available via PyTorch. This is a good memory-efficient alternative.")

In [ ]:
# Cell 15: WandB / TensorBoard logging setup

print("""
--- WandB Setup ---
To use Weights & Biases for experiment tracking:

  1. Install: !pip install wandb
  2. Login:   import wandb; wandb.login()
  3. In TrainingArguments, set: report_to="wandb"
  4. Optionally set: run_name="experiment-1"

WandB will automatically log:
  - Training/eval loss curves
  - Learning rate schedule
  - GPU utilization
  - Model gradients
  - System metrics (CPU, RAM, etc.)

--- TensorBoard Setup ---
To use TensorBoard:

  1. In TrainingArguments, set: report_to="tensorboard"
  2. Launch: %load_ext tensorboard
             %tensorboard --logdir {output_dir}/logs

TensorBoard will show the same loss curves locally.
""")

## Part 6: Troubleshooting Guide

Common issues and how to diagnose/fix them.

In [ ]:
# Cell 16: Troubleshooting — OOM (Out of Memory)

print("=" * 60)
print("TROUBLESHOOTING: Out of Memory (OOM)")
print("=" * 60)
print("""
If you hit CUDA Out of Memory, try these in order:

1. REDUCE BATCH SIZE
   per_device_train_batch_size=1  # Minimum

2. INCREASE GRADIENT ACCUMULATION
   gradient_accumulation_steps=8  # Keeps effective batch size

3. REDUCE SEQUENCE LENGTH
   max_seq_length=512  # or 256 for very constrained memory

4. ENABLE 4-BIT (already done if using QLoRA)
   load_in_4bit=True
   bnb_4bit_use_double_quant=True

5. ENABLE GRADIENT CHECKPOINTING
   gradient_checkpointing=True  # ~30% slower, ~50% less VRAM

6. USE CPU OFFLOADING (extreme case)
   from transformers import BitsAndBytesConfig
   # Add device_map with offload
   model = AutoModelForCausalLM.from_pretrained(
       ...,
       device_map="auto",
       offload_folder="offload",
   )

7. REDUCE LORA RANK
   r=4  # Minimum effective rank

8. CHECK FOR MEMORY LEAKS
   import gc; gc.collect()
   torch.cuda.empty_cache()
""")

In [ ]:
# Cell 17: Troubleshooting — Loss not converging

print("=" * 60)
print("TROUBLESHOOTING: Loss Not Converging")
print("=" * 60)
print("""
SYMPTOM: Loss stays flat or goes up.

POSSIBLE CAUSES & FIXES:

1. LEARNING RATE TOO HIGH
   -> Reduce: learning_rate=5e-5 (from 2e-4)
   -> For LoRA, 1e-4 to 5e-4 is typical; try the lower end

2. LEARNING RATE TOO LOW
   -> Increase: learning_rate=5e-4
   -> Loss barely moves each step

3. BAD DATA FORMATTING
   -> Verify the chat template is applied correctly
   -> Check that loss is only on assistant tokens
   -> Print a raw formatted example:
      print(dataset[0]['text'])

4. GRADIENT EXPLOSION
   -> Enable gradient clipping:
      max_grad_norm=1.0  # In TrainingArguments

5. INSUFFICIENT WARMUP
   -> Increase: warmup_ratio=0.1

6. DATA ISSUES
   -> Too few examples (< 100)
   -> Corrupted/binary text
   -> Mismatched tokenizer/model

7. WRONG OPTIMIZER
   -> LoRA works well with AdamW; avoid SGD
   -> optim="adamw_torch" or "paged_adamw_8bit"

DEBUGGING STEPS:
   a. Overfit on 10 examples first (should reach near-zero loss)
   b. Check tokenizer output for special tokens
   c. Verify data collator masking
   d. Try without quantization (load_in_4bit=False) to isolate
   e. Check for NaN in inputs/outputs:
      print(torch.isnan(loss).any())
""")

In [ ]:
# Cell 18: Troubleshooting — Check data quality mid-training

print("Diagnostic: Check your data pipeline")

# Verify tokenization
sample_text = train_data[0]["text"]
tokens = tokenizer.encode(sample_text)
print(f"Sample text length: {len(sample_text)} chars")
print(f"Tokenized length: {len(tokens)} tokens")
print(f"First 50 token IDs: {tokens[:50]}")

# Decode back to check for issues
decoded = tokenizer.decode(tokens[:100])
print(f"\nDecoded first 100 tokens:")
print(decoded)

# Check for excessive padding or truncation
all_lengths = [len(tokenizer.encode(ex["text"])) for ex in train_data.select(range(100))]
print(f"\nToken length stats (first 100 examples):")
print(f"  Min: {min(all_lengths)}, Max: {max(all_lengths)}")
print(f"  Mean: {np.mean(all_lengths):.0f}, Median: {np.median(all_lengths):.0f}")
print(f"  Examples > 1024 tokens: {sum(1 for l in all_lengths if l > 1024)}")

In [ ]:
# Cell 19: Summary

print("=" * 60)
print("NOTEBOOK 2 SUMMARY")
print("=" * 60)
print(f"""
We covered:
  - Loaded Qwen2.5-0.5B with NF4 4-bit quantization
  - Configured LoRA: r=16, alpha=32, target_modules=attention+MLP
  - Set up SFTTrainer with response-only loss masking
  - Trained with proper hyperparameters (lr=2e-4, cosine schedule...)
  - Plotted training/eval loss curves
  - Compared generation quality before vs after fine-tuning
  - Memory optimization techniques (4-bit, grad checkpointing, flash attn)
  - WandB / TensorBoard logging setup
  - Troubleshooting guides for OOM and convergence issues

Model artifacts saved:
  - LoRA adapter: ./qwen-lora-adapter/
  - Training outputs: ./qwen-lora-sft-output/

Next steps:
  - Notebook 3: DPO alignment on top of this SFT model
  - Notebook 4: Merge adapter and export for inference
""")